# Project: Exploring and Analyzing Movie Ratings with Polars



| File        | Column     | Description                                                                 |
|------------|-----------|------------------------------------------------------------------------------|
| movies.csv | movieId   | ID number that uniquely identifies each movie                                |
| movies.csv | title     | Name of the movie (may include year in parentheses)                          |
| movies.csv | genres    | Pipe-separated list of genres (e.g., Comedy|Drama)                           |
| ratings.csv| userId    | ID number that uniquely identifies each user                                 |
| ratings.csv| movieId   | ID number that uniquely identifies each movie                                |
| ratings.csv| rating    | Rating score from 0.5 to 5.0 (half-star increments allowed)                  |
| ratings.csv| timestamp | Unix timestamp when the rating was given                                     |
| tags.csv   | userId    | ID number that uniquely identifies each user                                 |
| tags.csv   | movieId   | ID number that uniquely identifies each movie                                |
| tags.csv   | tag       | User-generated tag (string, e.g., 'funny')                                    |
| tags.csv   | timestamp | Unix timestamp when the tag was added                                        |


In [ ]:
import polars as pl

In [10]:
movies = pl.read_csv("/content/movies.csv", separator=",")
ratings = pl.read_csv("/content/ratings.csv", separator=",")
tags = pl.read_csv("/content/tags.csv", separator=",")


print("--- Movies Sample ---")
print(movies.head(5))

print("\n--- Ratings Sample ---")
print(ratings.head(5))

print("\n--- Tags Sample ---")
print(tags.head(5))


print("\n--- Ratings Summary Statistics ---")
print(ratings.describe())

--- Movies Sample ---
shape: (5, 3)
┌─────────┬─────────────────────────────────┬─────────────────────────────────┐
│ movieId ┆ title                           ┆ genres                          │
│ ---     ┆ ---                             ┆ ---                             │
│ i64     ┆ str                             ┆ str                             │
╞═════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 1       ┆ Toy Story (1995)                ┆ Adventure|Animation|Children|C… │
│ 2       ┆ Jumanji (1995)                  ┆ Adventure|Children|Fantasy      │
│ 3       ┆ Grumpier Old Men (1995)         ┆ Comedy|Romance                  │
│ 4       ┆ Waiting to Exhale (1995)        ┆ Comedy|Drama|Romance            │
│ 5       ┆ Father of the Bride Part II (1… ┆ Comedy                          │
└─────────┴─────────────────────────────────┴─────────────────────────────────┘

--- Ratings Sample ---
shape: (5, 4)
┌────────┬─────────┬────────┬───────────┐
│ us

First, I loaded the movies.csv, ratings.csv, and tags.csv files using Polars with the standard comma separator.
After reading the datasets, I used .head() to inspect the top few rows of each table to verify their structure, column names, and formats. To understand the scale and basic statistics of the ratings dataset, I ran .describe(), which provided key summary measures like mean, count, and range.

## Step 2: Preprocess data

In [19]:
# Cast data types to standard integer formats
movies=movies.with_columns(pl.col("movieId").cast(pl.Int64))

ratings=ratings.with_columns([
  pl.col("userId").cast(pl.Int64),
  pl.col("movieId").cast(pl.Int64),
  pl.col("timestamp").cast(pl.Int64)
                              ])

tags=tags.with_columns([
  pl.col("userId").cast(pl.Int64),
  pl.col("movieId").cast(pl.Int64),
  pl.col("timestamp").cast(pl.Int64)
                              ])


In [20]:
# Check missing values count
print ("Missing values in movies:", movies.null_count())
print ("Missing values in ratings:", ratings.null_count())
print ("Missing values in tags:", tags.null_count())

Missing values in movies: shape: (1, 3)
┌─────────┬───────┬────────┐
│ movieId ┆ title ┆ genres │
│ ---     ┆ ---   ┆ ---    │
│ u32     ┆ u32   ┆ u32    │
╞═════════╪═══════╪════════╡
│ 0       ┆ 0     ┆ 0      │
└─────────┴───────┴────────┘
Missing values in ratings: shape: (1, 4)
┌────────┬─────────┬────────┬───────────┐
│ userId ┆ movieId ┆ rating ┆ timestamp │
│ ---    ┆ ---     ┆ ---    ┆ ---       │
│ u32    ┆ u32     ┆ u32    ┆ u32       │
╞════════╪═════════╪════════╪═══════════╡
│ 0      ┆ 0       ┆ 0      ┆ 0         │
└────────┴─────────┴────────┴───────────┘
Missing values in tags: shape: (1, 4)
┌────────┬─────────┬─────┬───────────┐
│ userId ┆ movieId ┆ tag ┆ timestamp │
│ ---    ┆ ---     ┆ --- ┆ ---       │
│ u32    ┆ u32     ┆ u32 ┆ u32       │
╞════════╪═════════╪═════╪═══════════╡
│ 0      ┆ 0       ┆ 0   ┆ 0         │
└────────┴─────────┴─────┴───────────┘


In [21]:
# Handle missing values (fill missing tag text with placeholder)
tags = tags.with_columns(pl.col("tag").fill_null("No Tag"))

In [22]:
# Check and remove duplicate rows
movies = movies.unique()
ratings = ratings.unique()
tags = tags.unique()

In [23]:
print("\nUpdated Movies Schema:")
print(movies.schema)


Updated Movies Schema:
Schema({'movieId': Int64, 'title': String, 'genres': String})



 I cast movieId, userId, and timestamp columns to integer format (pl.Int64) across all tables to keep data types consistent for merging later.
Using .null_count(), I found 0 missing values across all three datasets. Since the data is already complete, no row removal or filling was needed. This usually happens when system forms require mandatory user input before saving entries.
 I ran .unique() on the DataFrames to clean up any duplicate rows. Deduplication prevents repeated logs from distorting ratings or counts.

In [24]:
min_rating=ratings["rating"].min()

In [25]:
max_rating=ratings["rating"].max()

In [26]:
print("Minimum rating is:",min_rating )
print("Maximum rating is:",max_rating )


Minimum rating is: 0.5
Maximum rating is: 5.0


I checked minimum and maximum values show that data is sensible.

In [29]:
rating_counts=ratings.group_by('rating').count().sort('rating')
print(rating_counts)

shape: (10, 2)
┌────────┬───────┐
│ rating ┆ count │
│ ---    ┆ ---   │
│ f64    ┆ u32   │
╞════════╪═══════╡
│ 0.5    ┆ 1370  │
│ 1.0    ┆ 2811  │
│ 1.5    ┆ 1791  │
│ 2.0    ┆ 7551  │
│ 2.5    ┆ 5550  │
│ 3.0    ┆ 20047 │
│ 3.5    ┆ 13136 │
│ 4.0    ┆ 26818 │
│ 4.5    ┆ 8551  │
│ 5.0    ┆ 13211 │
└────────┴───────┘


/tmp/ipykernel_1435/2375327744.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  rating_counts=ratings.group_by('rating').count().sort('rating')


I computed the total number of ratings for each score from 0.5 to 5.0 and sorted the result to display the overall rating distribution in a clean table format.

In [30]:
df=movies.join(ratings, on='movieId').with_columns(pl.col('genres').str.split('|')).explode('genres')

In [32]:
avg_renge_rating=df.group_by('genres').agg(pl.col('rating').mean().alias('avg_rating')).sort('avg_rating', descending=True)

In [33]:
print(avg_renge_rating)

shape: (20, 2)
┌─────────────┬────────────┐
│ genres      ┆ avg_rating │
│ ---         ┆ ---        │
│ str         ┆ f64        │
╞═════════════╪════════════╡
│ Film-Noir   ┆ 3.920115   │
│ War         ┆ 3.808294   │
│ Documentary ┆ 3.797785   │
│ Crime       ┆ 3.658294   │
│ Drama       ┆ 3.656184   │
│ …           ┆ …          │
│ Sci-Fi      ┆ 3.455721   │
│ Action      ┆ 3.447984   │
│ Children    ┆ 3.412956   │
│ Comedy      ┆ 3.384721   │
│ Horror      ┆ 3.258195   │
└─────────────┴────────────┘


I joined the movies and ratings tables, split the pipe-separated genres, and expanded them into individual rows using .explode(). Then, I grouped the data by genre, calculated the average rating for each, and sorted the results in descending order.

In [35]:
user_counts=ratings.group_by('userId').len()

In [36]:
user_stats=user_counts.select(pl.col('len').min().alias('min'),
                              pl.col('len').max().alias('max'),
                              pl.col('len').mean().alias('mean'),
                              )

In [37]:
print(user_stats)

shape: (1, 3)
┌─────┬──────┬────────────┐
│ min ┆ max  ┆ mean       │
│ --- ┆ ---  ┆ ---        │
│ u32 ┆ u32  ┆ f64        │
╞═════╪══════╪════════════╡
│ 20  ┆ 2698 ┆ 165.304918 │
└─────┴──────┴────────────┘


I grouped the ratings table by userId to compute the total number of ratings given by each user. I then calculated the overall minimum, maximum, and average number of ratings per user across the dataset.

In [40]:
top5_users= ratings.group_by('userId').agg(
    pl.len().alias('rating_count')).sort('rating_count' , descending=True).head(5)

In [41]:
print('Top five users:' ,top5_users)

Top five users: shape: (5, 2)
┌────────┬──────────────┐
│ userId ┆ rating_count │
│ ---    ┆ ---          │
│ i64    ┆ u32          │
╞════════╪══════════════╡
│ 414    ┆ 2698         │
│ 599    ┆ 2478         │
│ 474    ┆ 2108         │
│ 448    ┆ 1864         │
│ 274    ┆ 1346         │
└────────┴──────────────┘


I grouped the ratings table by userId to count the total number of ratings per user, sorted the records in descending order, and extracted the top 5 most active users.

In [42]:
movie_stats= ratings.group_by('movieId').agg(pl.len().alias('count'),pl.col('rating').mean().alias('avg_rating')).filter(pl.col('count')>=5)

In [44]:
top10_avg=movie_stats.sort('count', descending=True).head(10).select(pl.col('avg_rating').mean()).item()
bottom10_avg=movie_stats.sort('count', descending=False).head(10).select(pl.col('avg_rating').mean()).item()

In [45]:
print(f"Top 10 most rated movies average rating: {top10_avg:.2f}")
print(f"Bottom 10 least rated movies average rating: {bottom10_avg:.2f}")

Top 10 most rated movies average rating: 4.14
Bottom 10 least rated movies average rating: 3.09


I calculated the total count and average rating for each movie, filtering out movies with fewer than 5 ratings. I then identified the top 10 most rated and bottom 10 least rated movies to compare their overall mean ratings.

In [51]:
movie_counts=ratings.group_by('movieId').agg(pl.len().alias('ratings_count'))

In [53]:
dist_table=movie_counts.group_by('ratings_count').agg(pl.len().alias("num_movies")).sort("ratings_count")


In [54]:
print(dist_table)

shape: (177, 2)
┌───────────────┬────────────┐
│ ratings_count ┆ num_movies │
│ ---           ┆ ---        │
│ u32           ┆ u32        │
╞═══════════════╪════════════╡
│ 1             ┆ 3446       │
│ 2             ┆ 1298       │
│ 3             ┆ 800        │
│ 4             ┆ 530        │
│ 5             ┆ 382        │
│ …             ┆ …          │
│ 278           ┆ 1          │
│ 279           ┆ 1          │
│ 307           ┆ 1          │
│ 317           ┆ 1          │
│ 329           ┆ 1          │
└───────────────┴────────────┘


I first grouped the dataset by movieId to determine how many ratings each movie received. Then, I grouped by those counts to build a distribution table showing the number of movies corresponding to each rating frequency, sorted in ascending order.

In [55]:
movie_counts=ratings.group_by('movieId').len().join(movies, on="movieId").rename({"len": "rating_count"})

In [56]:
top20_movies = movie_counts.sort("rating_count", descending=True).head(20).select(["movieId", "title", "rating_count"])

In [57]:
print(top20_movies)

shape: (20, 3)
┌─────────┬─────────────────────────────────┬──────────────┐
│ movieId ┆ title                           ┆ rating_count │
│ ---     ┆ ---                             ┆ ---          │
│ i64     ┆ str                             ┆ u32          │
╞═════════╪═════════════════════════════════╪══════════════╡
│ 356     ┆ Forrest Gump (1994)             ┆ 329          │
│ 318     ┆ Shawshank Redemption, The (199… ┆ 317          │
│ 296     ┆ Pulp Fiction (1994)             ┆ 307          │
│ 593     ┆ Silence of the Lambs, The (199… ┆ 279          │
│ 2571    ┆ Matrix, The (1999)              ┆ 278          │
│ …       ┆ …                               ┆ …            │
│ 47      ┆ Seven (a.k.a. Se7en) (1995)     ┆ 203          │
│ 780     ┆ Independence Day (a.k.a. ID4) … ┆ 202          │
│ 150     ┆ Apollo 13 (1995)                ┆ 201          │
│ 1198    ┆ Raiders of the Lost Ark (India… ┆ 200          │
│ 4993    ┆ Lord of the Rings: The Fellows… ┆ 198          │
└────────

I joined the ratings count with the movies dataframe to get the titles, sorted the movies by rating count in descending order, and printed the top 20 most frequently rated movies with their ID, title, and total count.

In [58]:
exploded_genres=movies.with_columns(pl.col('genres').str.split('|')).explode('genres')

In [59]:
unique_genres_count=exploded_genres['genres'].n_unique()
print(f"Total unique genres: {unique_genres_count}\n")

Total unique genres: 20



In [61]:
top10_genres = exploded_genres.group_by("genres").len().rename({"len": "count"}).sort("count", descending=True).head(10)
print(top10_genres)


shape: (10, 2)
┌───────────┬───────┐
│ genres    ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ Drama     ┆ 4361  │
│ Comedy    ┆ 3756  │
│ Thriller  ┆ 1894  │
│ Action    ┆ 1828  │
│ Romance   ┆ 1596  │
│ Adventure ┆ 1263  │
│ Crime     ┆ 1199  │
│ Sci-Fi    ┆ 980   │
│ Horror    ┆ 978   │
│ Fantasy   ┆ 779   │
└───────────┴───────┘


I split the pipe-separated genres column in the movies dataset and exploded it into individual rows. I then computed the total count of unique genres using .n_unique() and grouped the data by genre to identify the top 10 most frequent genres along with their counts.

In [62]:
exploded_movies = movies.with_columns(pl.col("genres").str.split("|")).explode("genres")

In [63]:
genre_stats = exploded_movies.join(ratings, on="movieId").group_by("genres").agg(pl.col("movieId").n_unique().alias("num_movies"), pl.col("rating").mean().alias("avg_rating")).sort("avg_rating", descending=True)

In [64]:
print(genre_stats)

shape: (20, 3)
┌─────────────┬────────────┬────────────┐
│ genres      ┆ num_movies ┆ avg_rating │
│ ---         ┆ ---        ┆ ---        │
│ str         ┆ u32        ┆ f64        │
╞═════════════╪════════════╪════════════╡
│ Film-Noir   ┆ 85         ┆ 3.920115   │
│ War         ┆ 381        ┆ 3.808294   │
│ Documentary ┆ 438        ┆ 3.797785   │
│ Crime       ┆ 1196       ┆ 3.658294   │
│ Drama       ┆ 4349       ┆ 3.656184   │
│ …           ┆ …          ┆ …          │
│ Sci-Fi      ┆ 980        ┆ 3.455721   │
│ Action      ┆ 1828       ┆ 3.447984   │
│ Children    ┆ 664        ┆ 3.412956   │
│ Comedy      ┆ 3753       ┆ 3.384721   │
│ Horror      ┆ 977        ┆ 3.258195   │
└─────────────┴────────────┴────────────┘


I split and exploded the genres column, merged the result with ratings, and grouped by genre to calculate both the unique movie count and the average rating, sorted in descending order by average rating.

In [65]:
tag_counts = tags.group_by("movieId").len()

In [66]:
tag_stats = tag_counts.select(pl.col("len").min().alias("min_tags"), pl.col("len").max().alias("max_tags"), pl.col("len").mean().alias("mean_tags"))

In [67]:
print(tag_stats)

shape: (1, 3)
┌──────────┬──────────┬───────────┐
│ min_tags ┆ max_tags ┆ mean_tags │
│ ---      ┆ ---      ┆ ---       │
│ u32      ┆ u32      ┆ f64       │
╞══════════╪══════════╪═══════════╡
│ 1        ┆ 181      ┆ 2.342875  │
└──────────┴──────────┴───────────┘


I grouped the tags dataset by movieId to calculate the total number of tags assigned to each movie. I then computed the overall minimum, maximum, and average number of tags per movie across the dataset.

In [68]:
top20_tags = tags.group_by("tag").len().rename({"len": "count"}).sort("count", descending=True).head(20)

In [69]:
print(top20_tags)

shape: (20, 2)
┌────────────────────┬───────┐
│ tag                ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ In Netflix queue   ┆ 131   │
│ atmospheric        ┆ 36    │
│ thought-provoking  ┆ 24    │
│ superhero          ┆ 24    │
│ surreal            ┆ 23    │
│ …                  ┆ …     │
│ visually appealing ┆ 19    │
│ politics           ┆ 18    │
│ mental illness     ┆ 16    │
│ time travel        ┆ 16    │
│ music              ┆ 16    │
└────────────────────┴───────┘


I grouped the tags dataset by the tag column to calculate how many times each tag was applied. Then, I sorted the counts in descending order and displayed the top 20 most frequent tags along with their total counts.

In [70]:
movie_high_ratings = ratings.group_by("movieId").agg(pl.len().alias("count"), (pl.col("rating") >= 4).mean().alias("high_rating_proportion")).filter(pl.col("count") >= 5)

In [71]:
result_table = movie_high_ratings.join(movies, on="movieId").select(["movieId", "title", "high_rating_proportion"]).sort("high_rating_proportion", descending=True)

In [72]:
print(result_table)

shape: (3_650, 3)
┌─────────┬─────────────────────────────────┬────────────────────────┐
│ movieId ┆ title                           ┆ high_rating_proportion │
│ ---     ┆ ---                             ┆ ---                    │
│ i64     ┆ str                             ┆ f64                    │
╞═════════╪═════════════════════════════════╪════════════════════════╡
│ 2239    ┆ Swept Away (Travolti da un ins… ┆ 1.0                    │
│ 106642  ┆ Day of the Doctor, The (2013)   ┆ 1.0                    │
│ 1192    ┆ Paris Is Burning (1990)         ┆ 1.0                    │
│ 3152    ┆ Last Picture Show, The (1971)   ┆ 1.0                    │
│ 96829   ┆ Hunt, The (Jagten) (2012)       ┆ 1.0                    │
│ …       ┆ …                               ┆ …                      │
│ 1646    ┆ RocketMan (a.k.a. Rocket Man) … ┆ 0.0                    │
│ 2163    ┆ Attack of the Killer Tomatoes!… ┆ 0.0                    │
│ 414     ┆ Air Up There, The (1994)        ┆ 0.0          

I grouped the ratings dataset by movieId to compute the total number of ratings and the proportion of ratings rated 4 or higher (rating >= 4). I filtered for movies with at least 5 ratings, joined with the movies dataset to fetch the titles, and displayed the final table sorted by high rating proportion in descending order.

In [73]:
user_reratings = ratings.group_by(["userId", "movieId"]).agg(pl.len().alias("c")).group_by("userId").agg(((pl.col("c") - 1).sum() / pl.col("c").sum()).alias("re_rating_proportion"))

In [74]:
top10_reraters = user_reratings.sort("re_rating_proportion", descending=True).head(10)

In [75]:
print(top10_reraters)

shape: (10, 2)
┌────────┬──────────────────────┐
│ userId ┆ re_rating_proportion │
│ ---    ┆ ---                  │
│ i64    ┆ f64                  │
╞════════╪══════════════════════╡
│ 153    ┆ 0.0                  │
│ 587    ┆ 0.0                  │
│ 385    ┆ 0.0                  │
│ 165    ┆ 0.0                  │
│ 314    ┆ 0.0                  │
│ 530    ┆ 0.0                  │
│ 328    ┆ 0.0                  │
│ 18     ┆ 0.0                  │
│ 87     ┆ 0.0                  │
│ 105    ┆ 0.0                  │
└────────┴──────────────────────┘


I grouped ratings by userId and movieId to detect multiple ratings per movie, calculated each user's proportion of re-ratings relative to their total ratings, and printed the top 10 users with the highest re-rating rates.

In [76]:
movie_stats = ratings.group_by("movieId").agg(pl.len().alias("count"), pl.col("rating").mean().alias("avg_rating")).filter(pl.col("count") >= 10).join(movies, on="movieId")

In [77]:
print(movie_stats.sort("avg_rating", descending=True).head(20).select(["movieId", "title", "avg_rating"]))

shape: (20, 3)
┌─────────┬─────────────────────────────────┬────────────┐
│ movieId ┆ title                           ┆ avg_rating │
│ ---     ┆ ---                             ┆ ---        │
│ i64     ┆ str                             ┆ f64        │
╞═════════╪═════════════════════════════════╪════════════╡
│ 1041    ┆ Secrets & Lies (1996)           ┆ 4.590909   │
│ 3451    ┆ Guess Who's Coming to Dinner (… ┆ 4.545455   │
│ 1178    ┆ Paths of Glory (1957)           ┆ 4.541667   │
│ 1104    ┆ Streetcar Named Desire, A (195… ┆ 4.475      │
│ 2360    ┆ Celebration, The (Festen) (199… ┆ 4.458333   │
│ …       ┆ …                               ┆ …          │
│ 7156    ┆ Fog of War: Eleven Lessons fro… ┆ 4.307692   │
│ 1209    ┆ Once Upon a Time in the West (… ┆ 4.305556   │
│ 92535   ┆ Louis C.K.: Live at the Beacon… ┆ 4.3        │
│ 96829   ┆ Hunt, The (Jagten) (2012)       ┆ 4.3        │
│ 55721   ┆ Elite Squad (Tropa de Elite) (… ┆ 4.3        │
└─────────┴──────────────────────────────

I calculated the rating count and average rating for each movie, filtered for those with at least 10 ratings, merged with the movies dataset to get the titles, and printed the top 20 movies sorted by average rating in descending order.

In [78]:
movie_stats = ratings.group_by("movieId").agg(pl.len().alias("count"), pl.col("rating").mean().alias("avg_rating"))

In [79]:
print(movie_stats.select(pl.corr("count", "avg_rating").alias("correlation")))

shape: (1, 1)
┌─────────────┐
│ correlation │
│ ---         │
│ f64         │
╞═════════════╡
│ 0.127259    │
└─────────────┘


I grouped the ratings dataset by movieId to compute the total rating count and average rating for each movie. Then, I calculated the Pearson correlation coefficient between the rating count and the average rating using pl.corr().